# SoloQ Pulse — Analysis Template

**Objective:** turn raw match data into an evidence-backed product analytics story.

> The dataset is synthetic. Do not present the results as real Riot/OP.GG/user data.


## 1. Setup & data loading
**TODO:** add any environment setup needed for your machine.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path('..').resolve()
RAW = BASE / 'data' / 'raw'

matches = pd.read_csv(RAW / 'matches.csv', parse_dates=['match_datetime'])
participants = pd.read_csv(RAW / 'match_participants.csv')
players = pd.read_csv(RAW / 'players.csv')
champions = pd.read_csv(RAW / 'champions.csv')
objectives = pd.read_csv(RAW / 'match_objectives.csv')

print(matches.shape, participants.shape, players.shape, champions.shape, objectives.shape)

## 2. Data quality
Run the supplied script first, then independently reproduce at least three checks here.

**TODO:** document the business consequence of any failed check.

In [ ]:
checks = {
    '10 participants per match': participants.groupby('match_id').size().eq(10).all(),
    'unique participant key': ~participants.duplicated(['match_id','player_id']).any(),
    'valid wins': participants['win'].isin([0,1]).all(),
}
checks

## 3. Feature engineering
Create derived variables that make the analysis business-readable.

**TODO:** KDA, CS/min, damage/min, vision/min, gold/min, month, and an objective metric.

In [ ]:
df = participants.merge(matches[['match_id','match_datetime','duration_min','patch']], on='match_id', how='left')
df = df.merge(players[['player_id','region','tier','main_role','play_style','current_mmr']], on='player_id', how='left')
df['month'] = df['match_datetime'].dt.to_period('M').astype(str)
df['kda'] = (df['kills'] + df['assists']) / df['deaths'].clip(lower=1)
df['cs_per_min'] = df['cs'] / df['duration_min']
df['damage_per_min'] = df['damage_to_champions'] / df['duration_min']
df['vision_per_min'] = df['vision_score'] / df['duration_min']
df['gold_per_min'] = df['gold_earned'] / df['duration_min']

df.head()

## 4. Executive KPIs
**TODO:** calculate and visualize at least five KPIs.

Recommended KPI questions:
- How much activity exists?
- What does an average match look like?
- Is performance stable over time?
- Where are the biggest differences between player groups?

In [ ]:
kpis = {
    'matches': matches['match_id'].nunique(),
    'players': players['player_id'].nunique(),
    'participant_rows': len(participants),
    'avg_duration_min': matches['duration_min'].mean(),
    'overall_win_rate': df['win'].mean(),
}
pd.Series(kpis)

## 5. Hypothesis 1 — What is associated with winning?
Test at least three candidate drivers: objective participation, CS/min, vision/min, KDA or lane advantage.

**TODO:** show effect size, sample size and an uncertainty measure where appropriate. Avoid causal wording.

In [ ]:
# TODO: bucket one metric and compare win rate by bucket.
# Example starter:
metric = 'objective_participation'
summary = df.groupby('win')[metric].agg(['count','mean','median'])
summary

## 6. Player segmentation
Build 3–5 descriptive segments from behavior, not from arbitrary labels.

Possible approach: aggregate player metrics, standardize selected variables, then use rule-based or clustering segmentation.

**TODO:** name each segment based on measured behavior and explain the business use of the segment.

In [ ]:
player_agg = df.groupby(['player_id','tier','region','main_role','play_style'], as_index=False).agg(
    matches=('match_id','nunique'),
    win_rate=('win','mean'),
    kda=('kda','mean'),
    cs_per_min=('cs_per_min','mean'),
    vision_per_min=('vision_per_min','mean'),
    objective_participation=('objective_participation','mean')
)
player_agg.head()

## 7. Champion intelligence
**TODO:** calculate pick volume, pick share and win rate. Apply a minimum sample threshold before making comparisons.

Questions:
- Which champions are frequently played?
- Which have high/low observed win rates?
- Does the pattern change by role or tier?
- Which champions appear consistent rather than noisy?

## 8. Engagement / activity
**TODO:** build a monthly activity table with active players, matches/player, repeat-player rate and an activity-change indicator.

In [ ]:
monthly = df.groupby('month').agg(
    active_players=('player_id','nunique'),
    matches=('match_id','nunique')
).reset_index()
monthly['matches_per_active_player'] = monthly['matches'] / monthly['active_players']
monthly

## 9. Statistical test
Choose one business-relevant comparison and test it. Example: objective-participation bands vs. win rate, or two play styles after stratifying by role.

**TODO:** state:
1. null hypothesis
2. test used
3. p-value / effect size
4. practical interpretation
5. limitation

## 10. Final story
Finish with a 5-slide executive storyline:

1. Situation / business question
2. What the data says
3. Why it matters
4. Recommended product experiment
5. KPI + measurement plan

The best submission leaves the reader with an explicit chain: **evidence → interpretation → action → measurement**.